# AM 11B Lab Activity 4: Consumer and Producer Surplus

**Week 6 — built on: Area Between Curves, Surpluses**

**Authors:** Julie Simons, with Claude (Sonnet 5, Anthropic), via Claude Code — September 2026

*With special thanks to Mariana Bessa and prior AM 11B students, whose earlier notebooks inspired this material, and to Eric Van Dusen and the [Data 88E](https://github.com/data-88e) course for inspiration.*

![Oranges for sale at a farmers market stall, with a price sign](https://thumb.wikimedia.org/wikipedia/commons/thumb/6/67/Fruit_and_vegetable_stall_at_Southside_Farmers_Market_November_2025.jpg/960px-Fruit_and_vegetable_stall_at_Southside_Farmers_Market_November_2025.jpg)

*Photo: Nick-D, CC BY-SA 4.0, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Fruit_and_vegetable_stall_at_Southside_Farmers_Market_November_2025.jpg)*

<!-- canvas-group-instructions -->
## Group Submission Instructions

**Joining your lab group on Canvas**

1. Agree with your partners on a group number (e.g., "Lab Group 7").
2. In our Canvas course, click **People**, then the **Groups** tab.
3. Find your group and click **Join**. **Every member must do this themselves.** You can't add your partners.
4. To check, click the group name and make sure all your partners are listed. If you joined the wrong group, click **Leave** and join the right one.
5. When you submit, **only one person uploads the notebook**. It counts for everyone in the group.

Please join your group before submitting your first lab. If you submit before joining, only you will get credit.

Please write the name(s) of your group (or yourself) below by double-clicking on this text.

**NAME(S):** *Replace this text with your name(s)*

## Learning Goals

By the end of this lab you should be able to:
1. Find market equilibrium price and quantity from demand and supply curves, by hand when the algebra is manageable.
2. Set up and evaluate the definite integrals for consumer surplus (CS) and producer surplus (PS), including by hand with the power rule for integration.
3. Interpret CS and PS as **areas between curves** — the same technique from this week's lecture.
4. Recognize that total surplus can be computed either as $CS+PS$ or as a single integral of $D(q)-S(q)$, and get the same answer either way.
5. See how surplus changes as market conditions (demand) shift.

### Why economists care
Consumer and producer surplus are the standard way economists measure the total benefit a market creates. They show up constantly in policy discussions — taxes, price controls, trade — as the thing we're trying to protect or the thing we're trading off.

## Setup

Run the cell below to load the tools we'll use today.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import random

sp.init_printing()
plt.rcParams['figure.figsize'] = (8, 6)
%matplotlib inline

def checkpoint(is_correct, hint=""):
    if is_correct:
        emoji = random.choice(["✅", "🚀", "⭐"])
        msg = random.choice([
            "Great job! You nailed it.",
            "You did it!",
            "Nailed it — nice work.",
            "Yes! That's exactly right.",
            "Correct! You're crushing this.",
        ])
        print(f"{emoji} {msg}")
    else:
        print("❌ Try again!" + (f" {hint}" if hint else ""))

# Answer checks live in lab_checks.py (in the same folder as this notebook)
try:
    from lab_checks import check, same_formula
except ImportError:
    def check(*args):
        print("ℹ️ Answer checking isn't available here because lab_checks.py isn't in the same folder "
              "as this notebook. Your answers are still saved.")

print("Setup complete!")

## Quick Review (Ungraded): The Gini Coefficient

Last week's lecture introduced the **Gini coefficient**, a single number summarizing income inequality in a population — built from an idea you already know well: *area under a curve*.

A **Lorenz curve** $L(x)$ plots the cumulative share of income earned by the poorest fraction $x$ of the population. If income were split perfectly equally, the Lorenz curve would just be the diagonal line $y=x$ — the poorest 20% of people would earn exactly 20% of income, and so on. In real economies, the Lorenz curve sags below that line, since the poorest fraction of the population earns less than its proportional share of income.

The Gini coefficient measures how far the Lorenz curve sags below the line of equality:

$$G = 1 - 2\int_0^1 L(x)\, dx$$

Geometrically, $G$ is twice the shaded area between the line of equality and the Lorenz curve. $G=0$ means perfect equality; $G$ close to $1$ means high inequality.

Run the cell below to see this for the example Lorenz curve $L(x) = x^2$ — nothing to edit, just run it and look at the picture. Today's lab reuses this exact "area under a curve" idea, applied to a new economic question: the surplus generated by a market.

*(This is a quick, ungraded warm-up — no need to record an answer.)*

In [ ]:
x = sp.symbols('x', positive=True)
L_expr = x**2   # example Lorenz curve: the poorest fraction x of the population earns x^2 of total income

G = 1 - 2*sp.integrate(L_expr, (x, 0, 1))
print(f"Gini coefficient for L(x) = x^2:  G = {G} = {float(G):.3f}")

x_vals = np.linspace(0, 1, 100)
L_vals = x_vals**2

plt.figure(figsize=(6, 6))
plt.plot(x_vals, x_vals, 'k--', label='Line of Equality')
plt.plot(x_vals, L_vals, 'b-', linewidth=2.5, label='Lorenz Curve $L(x)=x^2$')
plt.fill_between(x_vals, x_vals, L_vals, alpha=0.3, color='red', label='Area representing inequality')
plt.xlabel('Cumulative Share of Population')
plt.ylabel('Cumulative Share of Income')
plt.title(f'Lorenz Curve and Gini Coefficient (G = {float(G):.3f})')
plt.legend()
plt.axis('equal')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## The Scenario

A market has demand curve $D(q) = 100 - q^2$ and supply curve $S(q) = 3q^2$ (prices in dollars, quantity $q$ in hundreds of units).

Neither curve is a straight line this time:
- **Demand** is concave — willingness to pay starts at $100 when $q=0$ and falls off *faster and faster* as quantity rises (consumers get picky quickly).
- **Supply** is convex and starts at $0 — a classic **increasing marginal cost** story: the first units are nearly free to produce, but cost accelerates sharply as quantity grows.

This means CS and PS will be honest-to-goodness curved regions, not triangles — good practice for the area-between-curves technique from lecture, since you can't just use a triangle-area formula here.

### Step 1: Find the equilibrium — try it by hand first

At equilibrium, $D(q^*) = S(q^*)$. **Before running any code**, solve $100 - q^2 = 3q^2$ by hand for $q^*$ (basic algebra — no calculator needed, since $100/4=25$ has a nice square root), then plug back into either curve to find $p^*$. Type both values into the code cell below and run it.

</div>

In [ ]:
# EDIT HERE: replace each None with your hand-computed value
qstar_hand = None   # equilibrium quantity q*
pstar_hand = None   # equilibrium price p*

Now we'll verify your answers above. The cell below solves $D(q) = S(q)$ for $q^*$, plugs it back in to get $p^*$, and compares both with the values you entered. (We've told `sympy` that $q$ is positive, so it automatically discards the negative root of this quadratic equation.)

In [ ]:
q = sp.symbols('q', positive=True)
D = 100 - q**2
S = 3*q**2

qstar = sp.solve(sp.Eq(D, S), q)[0]
pstar = D.subs(q, qstar)

print(f"Equilibrium quantity: q* = {qstar} = {float(qstar):.3f}")
print(f"Equilibrium price:    p* = {pstar} = {float(pstar):.3f}")

print()

# Check your answer (the checking code lives in lab_checks.py)
check("lab4_equilibrium", qstar_hand, pstar_hand)

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

### Step 2: Consumer and producer surplus as areas between curves — try it by hand first

Recall from lecture:

$$CS = \int_0^{q^*} \big[D(q) - p^*\big]\, dq \qquad\qquad PS = \int_0^{q^*} \big[p^* - S(q)\big]\, dq$$

Each is literally an area between two curves: $D(q)$ and the horizontal line $p^*$ for CS, and $p^*$ and $S(q)$ for PS. The formulas don't care whether the curves are straight or not — that's the whole point of the technique.

**Before running any code**, use $q^*=5$ and $p^*=75$ from Step 1 to write out $D(q)-p^*$ and $p^*-S(q)$ as simple polynomials in $q$, then use the power rule for integration (antiderivative of $q^2$ is $q^3/3$) to evaluate both definite integrals by hand from $0$ to $5$. Type your values of CS and PS into the code cell below and run it.

</div>

In [ ]:
# EDIT HERE: replace each None with your hand-computed value
# (a fraction like 7/3 is fine, or a decimal rounded to 2 places)
CS_hand = None   # consumer surplus
PS_hand = None   # producer surplus

Now we'll verify your answers above. The cell below evaluates both integrals with `sympy` and compares the results with the values you entered.

In [ ]:
CS = sp.integrate(D - pstar, (q, 0, qstar))
PS = sp.integrate(pstar - S, (q, 0, qstar))

print(f"Consumer Surplus: {CS} = {float(CS):.3f}")
print(f"Producer Surplus: {PS} = {float(PS):.3f}")
print(f"Total Surplus:    {float(CS + PS):.3f}")

print()

# Check your answer (the checking code lives in lab_checks.py)
check("lab4_surplus", CS_hand, PS_hand)

## Visualizing the Surplus Regions

Run the cell below to see the demand and supply curves with CS shaded blue (above the price line, below demand) and PS shaded red (below the price line, above supply) — the same "area between curves" setup from lecture, just with curved boundaries instead of straight lines. (Demand is drawn solid and Supply dashed, and the two shaded regions get different hatch patterns too, so the plot still reads clearly even in black-and-white or for colorblind viewers.)

In [ ]:
q_vals = np.linspace(0, float(qstar)*1.3, 200)
D_vals = 100 - q_vals**2
S_vals = 3*q_vals**2

plt.figure()
plt.plot(q_vals, D_vals, 'b-', linewidth=2, label='Demand D(q)')
plt.plot(q_vals, S_vals, 'r--', linewidth=2, label='Supply S(q)')
plt.axhline(float(pstar), color='gray', linestyle='--', linewidth=1, label='Equilibrium price')
plt.axvline(float(qstar), color='gray', linestyle=':', linewidth=1)

q_fill = np.linspace(0, float(qstar), 100)
plt.fill_between(q_fill, 100 - q_fill**2, float(pstar), alpha=0.3, color='blue', hatch='//',
                  label='Consumer Surplus')
plt.fill_between(q_fill, float(pstar), 3*q_fill**2, alpha=0.3, color='red', hatch='..',
                  label='Producer Surplus')

plt.xlabel('Quantity (q)')
plt.ylabel('Price ($)')
plt.title('Market Equilibrium: Consumer & Producer Surplus')
plt.legend()
plt.show()

### Question 4.1: Reading the Areas

Unlike a straight-line example, here Producer Surplus ($250) is noticeably *larger* than Consumer Surplus ($83.33) — the opposite of what you might expect just by eyeballing the picture. Point to a specific feature of these two curves (think about where each one starts, at $q=0$, and how quickly it curves away from the equilibrium price) that explains why producers capture more surplus than consumers in this particular market.

*Replace this text with your answer.*

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

### Question 4.3 (Computational Check): Total Surplus as a Single Integral

Instead of computing $CS$ and $PS$ separately and adding them, total surplus can also be found directly as **one** definite integral of the *gap* between demand and supply:

$$TS = \int_0^{q^*} \big[D(q) - S(q)\big]\, dq$$

**Before running any code**, use $q^*=5$ to write $D(q)-S(q)$ as a single polynomial in $q$, then evaluate this integral by hand.

Which of the following is closest to your answer?

- A) \$83.33
- B) \$250.00
- C) \$333.33
- D) \$500.00

</div>

Enter your letter in the cell below and run it.

In [ ]:
mc_answer_4_3 = None   # EDIT HERE -- set to "A", "B", "C", or "D"

# Check your answer (the checking code lives in lab_checks.py)
check("lab4_q4_3", mc_answer_4_3)

<div style="background-color:#e7f1fc; border-left:6px solid #4a90d9; border-radius:4px; padding:4px 18px 4px 18px; color:#0b3d78; margin:12px 0;">

## Your Turn

Suppose consumer demand shifts to be **stronger**: a new quadratic demand curve $D_{new}(q) = a - b\,q^2$ with a higher starting willingness to pay at $q=0$ than the original $D(q)=100-q^2$. Pick your own values for `a_new` (the starting price) and `b_new` (how fast it curves down) below — supply stays the same, $S(q) = 3q^2$ — and the code will re-solve for the new equilibrium and new surplus values automatically.

</div>

In [ ]:
a_new = None   # EDIT HERE -- starting price of the new demand curve (pick something > 100)
b_new = None   # EDIT HERE -- how fast the new demand curve falls off
assert a_new is not None and b_new is not None, "Set a_new and b_new to numbers before running this cell!"

D_new = a_new - b_new*q**2
qstar_new = sp.solve(sp.Eq(D_new, S), q)[0]
pstar_new = D_new.subs(q, qstar_new)

CS_new = sp.integrate(D_new - pstar_new, (q, 0, qstar_new))
PS_new = sp.integrate(pstar_new - S, (q, 0, qstar_new))

print(f"New equilibrium: q* = {float(qstar_new):.3f}, p* = {float(pstar_new):.3f}")
print(f"New Consumer Surplus: {float(CS_new):.3f}  (was {float(CS):.3f})")
print(f"New Producer Surplus: {float(PS_new):.3f}  (was {float(PS):.3f})")
print(f"New Total Surplus:    {float(CS_new + PS_new):.3f}  (was {float(CS+PS):.3f})")

checkpoint(
    abs(float(D_new.subs(q, qstar_new)) - float(S.subs(q, qstar_new))) < 1e-6,
    hint="At your new equilibrium, D_new(q*) should exactly equal S(q*) -- make sure a_new and b_new got set."
)

Run the cell below to see the same picture redrawn for your new demand curve — same style as before: demand solid, supply dashed, CS and PS shaded with different hatch patterns.

In [ ]:
q_vals2 = np.linspace(0, float(qstar_new)*1.3, 200)
D_vals2 = a_new - b_new*q_vals2**2
S_vals2 = 3*q_vals2**2

plt.figure()
plt.plot(q_vals2, D_vals2, 'b-', linewidth=2, label='New Demand')
plt.plot(q_vals2, S_vals2, 'r--', linewidth=2, label='Supply')
plt.axhline(float(pstar_new), color='gray', linestyle='--', linewidth=1)

q_fill2 = np.linspace(0, float(qstar_new), 100)
plt.fill_between(q_fill2, a_new - b_new*q_fill2**2, float(pstar_new), alpha=0.3, color='blue', hatch='//', label='CS')
plt.fill_between(q_fill2, float(pstar_new), 3*q_fill2**2, alpha=0.3, color='red', hatch='..', label='PS')
plt.xlabel('Quantity (q)')
plt.ylabel('Price ($)')
plt.title('New Market Equilibrium')
plt.legend()
plt.show()

### Question 4.2: Total Surplus and Efficiency

Did total surplus (CS + PS) go up or down with the new demand curve? In one sentence, connect this to the idea that total surplus measures the overall benefit a market creates — was the market able to generate more total value with the new preferences?

*Replace this text with your answer.*

## Reflection

**In 1-2 sentences:** Why is it important that we integrate up to $q^*$ (the equilibrium quantity) and not some other quantity, when computing CS and PS?

*Replace this text with your answer.*

---

## Submission

1. Make sure you've run all cells in order (**Runtime/Kernel → Restart and run all** is a good final check).
2. Answer the short reflection question(s) above by editing the markdown cells.
3. Go to **File → Download** to save the `.ipynb` file to your computer.
4. Upload the file to Canvas under this Lab Activity.

Only one person per group needs to submit if you worked together — just make sure everyone's name is listed at the top.

**This lab is designed to take about 30 minutes.** If you're stuck for more than a few minutes on any one cell, that's your cue to ask a neighbor, a TA, or post on Ed Discussion — don't burn your whole discussion period on one line of code.
